# 🎙️ AI Voice Studio — Burmese Voice Clone (Kaggle GPU)

Run this notebook on Kaggle with **GPU** and **Internet ON**. VoxCPM2 supports Burmese and reference-audio voice cloning.

⚠️ This is a temporary free GPU backend. Keep the Kaggle session running while using the GitHub Pages app.

In [ ]:
!pip -q install voxcpm fastapi uvicorn python-multipart soundfile nest-asyncio
!apt-get -qq update && apt-get -qq install -y ffmpeg
print('Dependencies installed')

In [ ]:
import torch, os
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
else:
    raise RuntimeError('GPU is not enabled. In Kaggle Notebook Settings choose a GPU accelerator, then restart the session.')

In [ ]:
from huggingface_hub import snapshot_download
from voxcpm import VoxCPM
import soundfile as sf
print('Loading VoxCPM2... first run can take several minutes and download several GB.')
model = VoxCPM.from_pretrained('openbmb/VoxCPM2', load_denoiser=False)
print('VoxCPM2 loaded. Sample rate:', model.tts_model.sample_rate)

In [ ]:
import os, shutil, subprocess, tempfile
from pathlib import Path
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from starlette.background import BackgroundTask

app = FastAPI(title='AI Voice Studio Burmese Voice Clone')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=False, allow_methods=['*'], allow_headers=['*'])

def cleanup(path):
    try: os.remove(path)
    except FileNotFoundError: pass

def to_wav(src, dst):
    r = subprocess.run(['ffmpeg','-y','-i',src,'-ar','16000','-ac','1',dst], capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError('Audio conversion failed')

@app.get('/')
def root(): return {'ok': True, 'model': 'openbmb/VoxCPM2', 'language': 'Burmese (my)'}

@app.get('/health')
def health(): return {'ok': True, 'model': 'openbmb/VoxCPM2', 'language': 'Burmese (my)', 'voice_cloning': True}

@app.post('/clone')
async def clone(text: str = Form(...), audio: UploadFile = File(...)):
    text = text.strip()
    if not text: raise HTTPException(400, 'Text is required')
    if len(text) > 5000: raise HTTPException(400, 'Text must be 5000 characters or less')
    suffix = Path(audio.filename or 'sample.wav').suffix.lower() or '.wav'
    allowed = {'.wav','.mp3','.m4a','.flac','.ogg','.aac','.webm'}
    if suffix not in allowed: raise HTTPException(400, 'Unsupported audio format')
    data = await audio.read(25*1024*1024+1)
    if len(data) > 25*1024*1024: raise HTTPException(413, 'Audio file must be 25 MB or smaller')
    with tempfile.TemporaryDirectory() as td:
        src = os.path.join(td, 'input'+suffix); wav_path = os.path.join(td,'sample.wav'); out = os.path.join(td,'clone.wav')
        with open(src,'wb') as f: f.write(data)
        try:
            to_wav(src,wav_path)
            wav = model.generate(text=text, reference_wav_path=wav_path, cfg_value=2.0, inference_timesteps=10, max_len=600, normalize=False, denoise=False, retry_badcase=True, retry_badcase_max_times=2)
            sf.write(out,wav,model.tts_model.sample_rate)
        except Exception as e:
            raise HTTPException(500, f'Voice cloning failed: {e}')
        fd,persistent=tempfile.mkstemp(suffix='.wav'); os.close(fd)
        with open(out,'rb') as a, open(persistent,'wb') as b: b.write(a.read())
    return FileResponse(persistent, media_type='audio/wav', filename='burmese_voice_clone.wav', background=BackgroundTask(cleanup,persistent))

print('FastAPI app ready')

In [ ]:
# Start the API on port 7860
import nest_asyncio, threading, uvicorn
nest_asyncio.apply()
config = uvicorn.Config(app, host='0.0.0.0', port=7860, log_level='warning')
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
print('API running on http://127.0.0.1:7860')

In [ ]:
# Public temporary URL using Cloudflare Quick Tunnel
import subprocess, re, time, os, urllib.request
if not os.path.exists('/usr/local/bin/cloudflared'):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '/usr/local/bin/cloudflared')
    os.chmod('/usr/local/bin/cloudflared',0o755)
p = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(60):
    line = p.stdout.readline()
    if line:
        m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0); break
    time.sleep(0.5)
print('PUBLIC BACKEND URL:', public_url)
print('Paste that URL into the Backend URL box in your GitHub Pages Voice Clone page.')